# PatchTST Interpretability Analysis
Attention weights and channel importance for ASD aggression prediction.

In [1]:
import sys
sys.path.insert(0, '/home/ma.yun2/ASD_agg_deep_learning/shared')
sys.path.insert(0, '/home/ma.yun2/ASD_agg_deep_learning/models/patchtst')

import json
import pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

import wrappers
from networks.patchtst import build_patchtst_config, AggPatchTST

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

CHANNEL_NAMES = ['BVP', 'EDA', 'ACC_X', 'ACC_Y', 'ACC_Z', 'Magnitude', 'HR', 'RMSSD', 'PHASIC', 'TONIC']

Device: cpu


## 1. Load Trained Model

In [2]:
CHECKPOINT_DIR = '/home/ma.yun2/ASD_agg_deep_learning/experiments/results/patchtst/session_best_combined'
PARAMS_FILE    = f'{CHECKPOINT_DIR}/session_model_parameters.json'
WEIGHTS_FILE   = f'{CHECKPOINT_DIR}/session_model_patchtst_encoder.pth'

with open(PARAMS_FILE) as f:
    params = json.load(f)

classifier = wrappers.PatchTSTClassifier()
params['cuda'] = (DEVICE == 'cuda')
params['gpu']  = 0
classifier.set_params(**params)
classifier.load(f'{CHECKPOINT_DIR}/session_model')
classifier.model.eval()

print(f'Model loaded. Parameters: {classifier.model.count_parameters():,}')
print(f'  d_model={params["d_model"]}, n_layers={params["n_layers"]}, n_heads={params["n_heads"]}')
print(f'  patch_len={params["patch_len"]}, patch_stride={params["patch_stride"]}')

Model loaded. Parameters: 251,841
  d_model=64, n_layers=3, n_heads=8
  patch_len=32, patch_stride=16


## 2. Load Data Samples

In [3]:
# Load directly from cached .bin file — read-only, no writes to scratch
BIN_FILE = '/scratch/borasaniya.t/CBS_DATA_ASD_ONLY/dataInst_to180_tp180_mcFalse_rsFalse_bs15S.bin'

print('Loading cached instances from .bin file...')
with open(BIN_FILE, 'rb') as f:
    datalist = pickle.load(f)

dict_of_instances, dict_of_labels = datalist[0], datalist[1]

# Flatten all participants into arrays
all_X, all_y = [], []
for pid in dict_of_instances:
    inst = dict_of_instances[pid]   # [N, 10, 2880]
    lbl  = dict_of_labels[pid]      # [N]
    if len(inst) == 0:
        continue
    all_X.append(inst)
    all_y.append(lbl)

all_X = np.concatenate(all_X, axis=0).astype(np.float32)  # [total, 10, 2880]
all_y = np.concatenate(all_y, axis=0)

print(f'Total instances: {len(all_X)} | Positive: {all_y.sum()} | Negative: {(all_y==0).sum()}')

# Collect 50 positive and 50 negative samples
N_SAMPLES = 50
pos_idx = np.where(all_y == 1)[0][:N_SAMPLES]
neg_idx = np.where(all_y == 0)[0][:N_SAMPLES]

X_pos = torch.tensor(all_X[pos_idx]).to(DEVICE)  # [50, 10, 2880]
X_neg = torch.tensor(all_X[neg_idx]).to(DEVICE)
print(f'Positive samples: {X_pos.shape}, Negative samples: {X_neg.shape}')

Loading cached instances from .bin file...
Total instances: 107889 | Positive: 22015.0 | Negative: 85874
Positive samples: torch.Size([50, 10, 2880]), Negative samples: torch.Size([50, 10, 2880])


## 3. Extract Attention Weights

In [21]:
# Force eager attention at config level BEFORE running
classifier.model.model.config._attn_implementation = "eager"

# Verify
print("attn_implementation:", classifier.model.model.config._attn_implementation)

# Now patch eager_attention_forward in the patchtst namespace to force output_attentions=True
import transformers.models.patchtst.modeling_patchtst as patchtst_module

original_eager = patchtst_module.eager_attention_forward

def patched_eager(*args, **kwargs):
    kwargs['output_attentions'] = True
    return original_eager(*args, **kwargs)

patchtst_module.eager_attention_forward = patched_eager

# Now run
def get_attentions(model, X):
    attention_maps = []
    hooks = []

    def hook_fn(module, input, output):
        if isinstance(output, tuple) and output[1] is not None:
            attention_maps.append(output[1].detach().cpu())

    for name, module in model.model.named_modules():
        if type(module).__name__ == 'PatchTSTAttention':
            hooks.append(module.register_forward_hook(hook_fn))

    model.eval()
    with torch.no_grad():
        x_t = X.transpose(1, 2)
        model.model(past_values=x_t, output_attentions=True)

    for h in hooks:
        h.remove()

    return attention_maps

attn_pos = get_attentions(classifier.model, X_pos)
attn_neg = get_attentions(classifier.model, X_neg)

# Restore
patchtst_module.eager_attention_forward = original_eager
classifier.model.model.config._attn_implementation = "sdpa"

print(f'Captured {len(attn_pos)} layers')
print(f'Shape per layer: {attn_pos[0].shape}')

attn_implementation: eager
Captured 3 layers
Shape per layer: torch.Size([500, 8, 180, 180])


In [22]:
n_layers   = len(attn_pos)
n_channels = 10
batch_size = X_pos.shape[0]  # 50
n_patches  = attn_pos[0].shape[-1]  # 180

print(f'Attention layers:            {n_layers}')
print(f'Shape per layer:             {attn_pos[0].shape}')  # [500, 8, 180, 180]
print(f'Batch size:                  {batch_size}')
print(f'Channels:                    {n_channels}')
print(f'Patches per channel:         {n_patches}')

# Reshape to separate batch and channel dims for easier analysis
# [batch*channels, heads, patches, patches] -> [batch, channels, heads, patches, patches]
def reshape_attentions(attn_list, batch_size, n_channels):
    return [a.reshape(batch_size, n_channels, a.shape[1], a.shape[2], a.shape[3]) 
            for a in attn_list]

attn_pos_r = reshape_attentions(attn_pos, batch_size, n_channels)
attn_neg_r = reshape_attentions(attn_neg, batch_size, n_channels)
print(f'Reshaped: {attn_pos_r[0].shape}')  # [50, 10, 8, 180, 180]

Attention layers:            3
Shape per layer:             torch.Size([500, 8, 180, 180])
Batch size:                  50
Channels:                    10
Patches per channel:         180
Reshaped: torch.Size([50, 10, 8, 180, 180])


## 4. CLS Token Attention — Which Patches Matter Most?

In [23]:
def cls_attention_map(attentions, batch_size, n_channels, last_layer_only=True):
    """
    Extract CLS token attention over patches.
    Returns: [batch, n_channels, n_patches] averaged over heads
    """
    layers = [attentions[-1]] if last_layer_only else attentions
    attn_maps = []

    for layer_attn in layers:
        # layer_attn: [batch*channels, heads, patches+1, patches+1]
        # CLS is index 0 — attention from CLS to all patches
        cls_attn = layer_attn[:, :, 0, 1:]  # [batch*ch, heads, patches]
        cls_attn = cls_attn.mean(dim=1)     # avg over heads: [batch*ch, patches]
        cls_attn = cls_attn.view(batch_size, n_channels, -1)  # [batch, ch, patches]
        attn_maps.append(cls_attn)

    return torch.stack(attn_maps).mean(dim=0)  # avg over layers

cls_pos = cls_attention_map(attn_pos, batch_size, n_channels).cpu().numpy()  # [50, 10, n_patches]
cls_neg = cls_attention_map(attn_neg, batch_size, n_channels).cpu().numpy()

# Average over samples
mean_attn_pos = cls_pos.mean(axis=0)  # [10, n_patches]
mean_attn_neg = cls_neg.mean(axis=0)

print(f'Attention map shape: {mean_attn_pos.shape} (channels x patches)')

Attention map shape: (10, 179) (channels x patches)


In [24]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, data, title in zip(axes, [mean_attn_pos, mean_attn_neg],
                           ['Aggression (Positive)', 'No Aggression (Negative)']):
    im = ax.imshow(data, aspect='auto', cmap='hot', interpolation='nearest')
    ax.set_yticks(range(n_channels))
    ax.set_yticklabels(CHANNEL_NAMES)
    ax.set_xlabel('Patch index (early → late in window)')
    ax.set_title(f'CLS Attention Heatmap — {title}')
    plt.colorbar(im, ax=ax, label='Attention weight')

plt.tight_layout()
plt.savefig('attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: attention_heatmap.png')

Saved: attention_heatmap.png


## 5. Channel Importance — Which Channels Are Most Attended To?

In [25]:
# Sum attention over patches per channel → channel importance score
channel_importance_pos = mean_attn_pos.sum(axis=1)  # [10]
channel_importance_neg = mean_attn_neg.sum(axis=1)

# Normalize
channel_importance_pos /= channel_importance_pos.sum()
channel_importance_neg /= channel_importance_neg.sum()

x = np.arange(n_channels)
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, channel_importance_pos, width, label='Aggression', color='tomato', alpha=0.8)
bars2 = ax.bar(x + width/2, channel_importance_neg, width, label='No Aggression', color='steelblue', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(CHANNEL_NAMES, rotation=30, ha='right')
ax.set_ylabel('Normalized attention weight')
ax.set_title('Channel Importance: Aggression vs No Aggression')
ax.legend()
plt.tight_layout()
plt.savefig('channel_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: channel_importance.png')

Saved: channel_importance.png


## 6. Temporal Attention — Early vs Late Patches

In [26]:
# Average attention over all channels → temporal importance
temporal_pos = mean_attn_pos.mean(axis=0)  # [n_patches]
temporal_neg = mean_attn_neg.mean(axis=0)

patch_len    = params['patch_len']
patch_stride = params['patch_stride']
fs           = 16  # Hz
patch_times  = [(i * patch_stride) / fs for i in range(len(temporal_pos))]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(patch_times, temporal_pos, color='tomato',    label='Aggression',    linewidth=2)
ax.plot(patch_times, temporal_neg, color='steelblue', label='No Aggression', linewidth=2)
ax.set_xlabel('Patch start time (seconds into window)')
ax.set_ylabel('Mean attention weight')
ax.set_title('Temporal Attention Profile — Which part of the 3-min window matters?')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('temporal_attention.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: temporal_attention.png')

Saved: temporal_attention.png


## 7. Attention Difference Map (Aggression − No Aggression)

In [27]:
diff_map = mean_attn_pos - mean_attn_neg  # [10, n_patches]
vmax = np.abs(diff_map).max()

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(diff_map, aspect='auto', cmap='RdBu_r',
               vmin=-vmax, vmax=vmax, interpolation='nearest')
ax.set_yticks(range(n_channels))
ax.set_yticklabels(CHANNEL_NAMES)
ax.set_xlabel('Patch index')
ax.set_title('Attention Difference Map (Aggression − No Aggression)\nRed = more attended during aggression, Blue = more attended during no aggression')
plt.colorbar(im, ax=ax, label='Attention difference')
plt.tight_layout()
plt.savefig('attention_diff_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: attention_diff_map.png')

Saved: attention_diff_map.png


## 8. Per-Layer Attention (How Attention Evolves Across Layers)

In [28]:
fig, axes = plt.subplots(1, n_layers, figsize=(6 * n_layers, 5))
if n_layers == 1:
    axes = [axes]

for layer_idx, ax in enumerate(axes):
    layer_attn = attn_pos[layer_idx]           # [batch*ch, heads, patches+1, patches+1]
    cls_attn   = layer_attn[:, :, 0, 1:]       # CLS → patches
    cls_attn   = cls_attn.mean(dim=1)          # avg heads
    cls_attn   = cls_attn.view(batch_size, n_channels, -1).mean(dim=0).cpu().numpy()  # [ch, patches]

    im = ax.imshow(cls_attn, aspect='auto', cmap='hot', interpolation='nearest')
    ax.set_yticks(range(n_channels))
    ax.set_yticklabels(CHANNEL_NAMES if layer_idx == 0 else [])
    ax.set_title(f'Layer {layer_idx + 1}')
    ax.set_xlabel('Patch')
    plt.colorbar(im, ax=ax)

fig.suptitle('CLS Attention Per Layer (Aggression samples)', fontsize=13)
plt.tight_layout()
plt.savefig('attention_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: attention_per_layer.png')

Saved: attention_per_layer.png


## 9. Summary Statistics

In [29]:
print('=== Channel Importance Ranking (Aggression samples) ===')
ranked = np.argsort(channel_importance_pos)[::-1]
for rank, ch_idx in enumerate(ranked):
    print(f'  {rank+1}. {CHANNEL_NAMES[ch_idx]:<12} {channel_importance_pos[ch_idx]:.4f}')

print('\n=== Temporal Focus ===')
peak_patch_pos = np.argmax(temporal_pos)
peak_patch_neg = np.argmax(temporal_neg)
print(f'  Aggression peak attention:    patch {peak_patch_pos} ({patch_times[peak_patch_pos]:.1f}s)')
print(f'  No aggression peak attention: patch {peak_patch_neg} ({patch_times[peak_patch_neg]:.1f}s)')

print('\n=== Top Channel-Patch Combinations (Aggression) ===')
flat_idx = np.argsort(mean_attn_pos.ravel())[::-1][:10]
for idx in flat_idx:
    ch, patch = divmod(idx, mean_attn_pos.shape[1])
    print(f'  {CHANNEL_NAMES[ch]:<12} patch {patch:3d} ({patch_times[patch]:.1f}s)  attn={mean_attn_pos[ch, patch]:.4f}')

=== Channel Importance Ranking (Aggression samples) ===
  1. TONIC        0.1001
  2. EDA          0.1001
  3. ACC_X        0.1001
  4. PHASIC       0.1001
  5. ACC_Z        0.1000
  6. Magnitude    0.1000
  7. ACC_Y        0.1000
  8. HR           0.0999
  9. RMSSD        0.0999
  10. BVP          0.0998

=== Temporal Focus ===
  Aggression peak attention:    patch 106 (106.0s)
  No aggression peak attention: patch 74 (74.0s)

=== Top Channel-Patch Combinations (Aggression) ===
  RMSSD        patch  75 (75.0s)  attn=0.0066
  RMSSD        patch  83 (83.0s)  attn=0.0066
  RMSSD        patch  82 (82.0s)  attn=0.0065
  RMSSD        patch  90 (90.0s)  attn=0.0065
  RMSSD        patch  67 (67.0s)  attn=0.0065
  RMSSD        patch  74 (74.0s)  attn=0.0065
  RMSSD        patch  98 (98.0s)  attn=0.0065
  RMSSD        patch 100 (100.0s)  attn=0.0065
  RMSSD        patch  81 (81.0s)  attn=0.0065
  RMSSD        patch 115 (115.0s)  attn=0.0064
